In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip /content/data.zip


# STEP 1. data.yaml 생성
import yaml, os, shutil, random
from glob import glob
from tqdm import tqdm

yaml_data = {
    'train': './data/images/train',
    'val': './data/images/val',
    'test': './data/images/test',
    'nc': 4,
    'names': ['Pothole', 'Alligator Crack', 'Transverse Crack', 'Longitudinal Crack']
}
with open('data.yaml', 'w') as f:
    yaml.dump(yaml_data, f, sort_keys=False)

print("✅ data.yaml 생성 완료")

# STEP 2. 데이터셋 병합 및 분할
img_dirs = ['country_1/images', 'country_2/images', 'country_3/images']
lbl_dirs = ['country_1/labels', 'country_2/labels', 'country_3/labels']

splits = ['train', 'val', 'test']
base_dir = 'data'
for split in splits:
    os.makedirs(f'{base_dir}/images/{split}', exist_ok=True)
    os.makedirs(f'{base_dir}/labels/{split}', exist_ok=True)

all_pairs = []
for img_dir, lbl_dir in zip(img_dirs, lbl_dirs):
    imgs = sorted(glob(f'{img_dir}/*.jpg')) + sorted(glob(f'{img_dir}/*.png'))
    for img in imgs:
        base = os.path.basename(img)
        lbl = os.path.join(lbl_dir, base.replace('.jpg', '.txt').replace('.png', '.txt'))
        if os.path.exists(lbl):
            all_pairs.append((img, lbl))

random.seed(42)
random.shuffle(all_pairs)
n = len(all_pairs)
train_end, val_end = int(0.7 * n), int(0.85 * n)
split_data = {
    'train': all_pairs[:train_end],
    'val': all_pairs[train_end:val_end],
    'test': all_pairs[val_end:]
}

for split, pairs in split_data.items():
    for img, lbl in tqdm(pairs, desc=f'Copying {split}'):
        shutil.copy(img, f'{base_dir}/images/{split}/{os.path.basename(img)}')
        shutil.copy(lbl, f'{base_dir}/labels/{split}/{os.path.basename(lbl)}')

print("✅ 데이터셋 준비 완료")

# STEP 3. YOLOv8 학습
!pip install ultralytics
!pip install --upgrade ultralytics

from ultralytics import YOLO


model = YOLO('yolo11m.pt')
model.train(data='data.yaml', epochs=50, imgsz=640, batch=16)

# STEP 4. 테스트 평가 (F1, mAP 등)
results = model.val(data='data.yaml', split='test', imgsz=640)


print(f"📊 Precision: {results.box.precision:.4f}")
print(f"📊 Recall: {results.box.recall:.4f}")
print(f"📊 mAP@0.5: {results.box.map50:.4f}")
print(f"📊 mAP@0.5:0.95: {results.box.map:.4f}")

# STEP 5. best.pt Google Drive 업로드
train_dirs = sorted(glob('runs/detect/train*'), key=os.path.getmtime, reverse=True)
latest_train = train_dirs[0]
best_path = os.path.join(latest_train, 'weights', 'best.pt')

dest = '/content/drive/MyDrive/yolo_models/best.pt'
os.makedirs(os.path.dirname(dest), exist_ok=True)
shutil.copy(best_path, dest)

print(f"✅ 모델이 Google Drive에 저장되었습니다 → {dest}")


In [ ]:
results = model.val(data="data.yaml")

# 결과 객체 속성 출력
print("🔍 results 속성 목록:", dir(results))
print("📦 box 속성 내용:")
print(vars(results.box))  # 혹은 dir(results.box)



In [ ]:
from ultralytics import YOLO
import pandas as pd

# 1. 모델 로드
model = YOLO('/content/runs/detect/train/weights/best.pt')  # 또는 best.pt, 경로에 따라 수정

# 2. 검증 실행
results = model.val(data="data.yaml")  # 또는 custom YAML 경로

# 3. 클래스 이름 및 평가 지표 불러오기
class_names = results.names  # dict: {0: 'Pothole', 1: 'Alligator Crack', ...}
precisions = results.box.p
recalls = results.box.r
f1_scores = results.box.f1

# 4. 출력
print("📊 F1-score per class:")
for i in range(len(f1_scores)):
    print(f"{class_names[i]}: F1-score = {f1_scores[i]:.4f} (P={precisions[i]:.3f}, R={recalls[i]:.3f})")

In [ ]:
from ultralytics import YOLO
import pandas as pd

# 1. 하이퍼파라미터를 dict로 정의
hyp_dict = {
    "lr0": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "degrees": 0.2,
    "translate": 0.1,
    "scale": 0.5,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "mixup": 0.2,
    "copy_paste": 0.0
}

# 2. 모델 로드 (fine-tuning용 best.pt)
model = YOLO("/content/runs/detect/train/weights/best.pt")

# 3. 학습 시작 (하이퍼파라미터는 dict로 전달)
model.train(
    data="/content/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="yolo11m_augmented_v3",
    resume=False,
    **hyp_dict  # ✅ dict를 unpack하여 전달!
)


results = model.val(data="/content/data.yaml")

class_names = results.names
precisions = results.box.p
recalls = results.box.r
f1_scores = results.box.f1

for i in range(len(f1_scores)):
    print(f"{class_names[i]}: F1-score = {f1_scores[i]:.4f} (P={precisions[i]:.3f}, R={recalls[i]:.3f})")

df = pd.DataFrame({
    "Class": [class_names[i] for i in range(len(f1_scores))],
    "Precision": precisions,
    "Recall": recalls,
    "F1-score": f1_scores
})
df.to_csv("/content/f1_scores_augmented.csv", index=False)


In [ ]:
from ultralytics import YOLO

# 이전 best.pt 로드
model = YOLO('runs/detect/train/weights/best.pt')

# 추가로 50 에폭 학습 (기존 50 + 추가 50 = 총 100)
model.train(data='data.yaml', epochs=100, imgsz=640, batch=16)

In [ ]:
from ultralytics import YOLO
import pandas as pd

# 1. 모델 로드
model = YOLO('/content/runs/detect/train/weights/best.pt')  # 또는 best.pt, 경로에 따라 수정

# 2. 검증 실행
results = model.val(data="data.yaml")  # 또는 custom YAML 경로

# 3. 클래스 이름 및 평가 지표 불러오기
class_names = results.names  # dict: {0: 'Pothole', 1: 'Alligator Crack', ...}
precisions = results.box.p
recalls = results.box.r
f1_scores = results.box.f1

# 4. 출력
print("📊 F1-score per class:")
for i in range(len(f1_scores)):
    print(f"{class_names[i]}: F1-score = {f1_scores[i]:.4f} (P={precisions[i]:.3f}, R={recalls[i]:.3f})")

In [ ]:
# 📦 1. 필요한 라이브러리 설치
!pip install -q albumentations opencv-python

# 📁 2. 폴더 구조 생성
import os

base_img_dir = '/content/data/images'
base_lbl_dir = '/content/data/labels'

os.makedirs(f'{base_img_dir}/train_aug', exist_ok=True)
os.makedirs(f'{base_lbl_dir}/train_aug', exist_ok=True)

os.makedirs(f'{base_img_dir}/train', exist_ok=True)  # 원본 train 폴더 없을 수도 있음
os.makedirs(f'{base_lbl_dir}/train', exist_ok=True)

# 🧠 3. 증강 함수 정의
import cv2
import numpy as np
from glob import glob
import albumentations as A
from tqdm import tqdm
import shutil

def yolo_to_voc(bbox, img_width, img_height):
    x, y, w, h = bbox
    x_min = (x - w / 2) * img_width
    y_min = (y - h / 2) * img_height
    x_max = (x + w / 2) * img_width
    y_max = (y + h / 2) * img_height
    return [x_min, y_min, x_max, y_max]

def voc_to_yolo(bbox, img_width, img_height):
    x_min, y_min, x_max, y_max = bbox
    x = ((x_min + x_max) / 2) / img_width
    y = ((y_min + y_max) / 2) / img_height
    w = (x_max - x_min) / img_width
    h = (y_max - y_min) / img_height
    return [x, y, w, h]

def read_yolo_labels(label_path):
    with open(label_path, 'r') as f:
        lines = f.readlines()
    labels = []
    for line in lines:
        cls, x, y, w, h = map(float, line.strip().split())
        labels.append([cls, x, y, w, h])
    return labels

def save_yolo_labels(labels, save_path):
    with open(save_path, 'w') as f:
        for label in labels:
            f.write(" ".join(map(str, label)) + '\n')

# 📈 4. 증강 transform 정의
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.2, rotate_limit=10, p=0.7),
    A.Blur(blur_limit=3, p=0.2),
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['category_ids']))

# 🖼️ 5. 증강 실행
image_paths = glob(f'{base_img_dir}/train/*.jpg') + glob(f'{base_img_dir}/train/*.png')

print(f'총 {len(image_paths)}개의 이미지에 대해 증강 시작...')

for img_path in tqdm(image_paths):
    img = cv2.imread(img_path)
    if img is None:
        continue
    h, w = img.shape[:2]
    base = os.path.basename(img_path).rsplit('.', 1)[0]
    label_path = f'{base_lbl_dir}/train/{base}.txt'

    if not os.path.exists(label_path):
        continue

    labels = read_yolo_labels(label_path)
    bboxes = [yolo_to_voc(label[1:], w, h) for label in labels]
    classes = [int(label[0]) for label in labels]

    for i in range(2):  # 이미지당 2개씩 증강
        transformed = transform(image=img, bboxes=bboxes, category_ids=classes)
        new_img = transformed['image']
        new_bboxes = transformed['bboxes']
        new_classes = transformed['category_ids']

        yolo_labels = []
        for cls, bbox in zip(new_classes, new_bboxes):
            yolo_bbox = voc_to_yolo(bbox, w, h)
            yolo_labels.append([cls] + yolo_bbox)

        # 저장
        new_img_path = f'{base_img_dir}/train_aug/{base}_aug{i}.jpg'
        new_lbl_path = f'{base_lbl_dir}/train_aug/{base}_aug{i}.txt'

        cv2.imwrite(new_img_path, new_img)
        save_yolo_labels(yolo_labels, new_lbl_path)

print("✅ 증강 완료!")

# 🔁 6. 증강된 데이터 train에 병합
aug_img_paths = glob(f'{base_img_dir}/train_aug/*')
aug_lbl_paths = glob(f'{base_lbl_dir}/train_aug/*')

print("🔄 증강본을 train 폴더로 병합 중...")

for p in aug_img_paths:
    shutil.copy(p, f'{base_img_dir}/train/')
for p in aug_lbl_paths:
    shutil.copy(p, f'{base_lbl_dir}/train/')

print("🎉 모든 증강본 병합 완료!")


In [ ]:
from ultralytics import YOLO

model = YOLO('/content/runs/detect/train/weights/best.pt')
model.train(data="data.yaml", epochs=50, imgsz=640, batch=16)